<div dir="rtl" align="right">

# إزالةُ ضوضاءِ شبكةِ الكهرباءِ بِـ CCA

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نَستخدمُ التحليلَ الارتباطيَّ القانونيَّ (CCA) لِإيجادِ المكوّنِ في إشارةِ EEG الذي يَرتبطُ بِضوضاءِ شبكةِ الكهرباءِ (50 Hz)، ثمّ نَطرحُهُ من الإشارةِ.

## المُخرجاتُ المُتوقّعةُ

- الإشارةُ الأصليّةُ في الأعلى
- الإشارةُ المُنظّفةُ بالأسفل معَ المكوّنِ المُزالِ باللونِ الأحمر
- اختفاءُ التذبذبِ المُنتظمِ عندَ 50 Hz

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| الترددُ | 50 Hz | ترددُ شبكةِ الكهرباءِ |
| n_components | 1 | مكوّنٌ واحدٌ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ CCA لِإزالةِ الضوضاءِ

نُنشئُ إشارةً مرجعيّةً عندَ 50 Hz (جيب وجيب تمام)، ثمّ نُطبّقُ CCA لِإيجادِ المكوّنِ المرتبطِ ونَطرحُهُ.

</div>

In [ ]:
from sklearn.cross_decomposition import CCA

channel_data = eeg_data[:, 0]
t = np.arange(len(channel_data)) / fs
ref_sin = np.sin(2 * np.pi * 50.0 * t).reshape(-1, 1)
ref_cos = np.cos(2 * np.pi * 50.0 * t).reshape(-1, 1)
reference = np.hstack([ref_sin, ref_cos])

eeg_2d = channel_data.reshape(-1, 1)
cca = CCA(n_components=1)
cca.fit(eeg_2d, reference)
eeg_c, ref_c = cca.transform(eeg_2d, reference)

artifact = eeg_c[:, 0]
artifact = artifact / artifact.std() * channel_data.std()
cleaned = channel_data - artifact
print(f'Artifact std: {artifact.std():.2f}, Cleaned std: {cleaned.std():.2f}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- المكوّنُ الأحمرُ يَلتقطُ التذبذبَ عندَ 50 Hz
- الإشارةُ الخضراءُ هي الإشارةُ بعدَ إزالةِ الضوضاءِ
- استخدمْ التكبيرَ لِملاحظةِ الفرقِ الدقيق


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = np.arange(n_plot) / fs

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original signal - Channel P4',
                                    'CCA Cleaned Signal (50 Hz removed)'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Original',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=cleaned[:n_plot], name='Cleaned',
                         line=dict(color='green', width=0.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=artifact[:n_plot], name='Removed',
                         line=dict(color='red', width=1)), row=2, col=1)
fig.update_layout(height=700, title_text='CCA Artifact Removal - Powerline (50 Hz)',
                  xaxis2_title='Time (s)', showlegend=True)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- CCA يَبحثُ عن أعلى ارتباطٍ بينَ مجموعتين من المتغيّرات
- نَستخدمُ إشارةً مرجعيّةً لِتمثيلِ الأثرِ الشائب
- يُزيلُ المكوّنَ المرتبطَ دونَ التأثيرِ على المحتوى الدماغيِّ
- فعّالٌ لِإزالةِ ضوضاءِ شبكةِ الكهرباءِ


</div>